In [60]:
!pip3 install beautifulsoup4 requests

In [61]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import sqlite3

#Task-1
Using the requests and BeautifulSoup libraries, scrape all books listed across at least 3 different book categories (or, if you prefer, the first 5 paginated listing pages of the "All products" catalogue — either scope is acceptable as long as your final dataset has at least 60 books). For each book capture: title, price (as listed, in GBP), star_rating (as text, e.g. "Three"), availability (as listed text), and category.

In [62]:
#Task-1

category_elements=[
      {"name":"Mystery","url":"https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"},
      {"name":"Historical Fiction","url":"https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html"},
      {"name":"Sequential Art","url":"https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html"},
      {"name":"Fiction","url":"https://books.toscrape.com/catalogue/category/books/fiction_10/index.html"}
  ]
books_details=[]
#print(category_elements['url'])

for cat in category_elements:
  #print(cat['url'])
  response=requests.get(cat["url"])
  if response.status_code == 200:
    #print(response.status_code)
    soup=BeautifulSoup(response.content,'html.parser')
    books_products=soup.find_all('article','product_pod')

    for book in books_products[:]:
      title=book.find("h3").find("a")["title"]
      price=book.find("p",class_="price_color").text.strip()

      for c in book.find("p",class_="star-rating")["class"]:
        if c!="star-rating":
          star_rating=c
      availability=book.find("p",class_="instock availability").text.strip()
      category=cat["name"]

      books_details.append({
          "title":title,
          "price":price,
          "star_rating":star_rating,
          "availability":availability,
          "category":category
      })
      #print(title,'-',price,'-',availability,'-',category,'-',star_rating)

  else:
    print(f"Failed to retrive. Status Code:{response.status_code}")

print(f"Total number of books retrived:{len(books_details)}")

books_details

Total number of books retrived:80


[{'title': 'Sharp Objects',
  'price': '£47.82',
  'star_rating': 'Four',
  'availability': 'In stock',
  'category': 'Mystery'},
 {'title': 'In a Dark, Dark Wood',
  'price': '£19.63',
  'star_rating': 'One',
  'availability': 'In stock',
  'category': 'Mystery'},
 {'title': 'The Past Never Ends',
  'price': '£56.50',
  'star_rating': 'Four',
  'availability': 'In stock',
  'category': 'Mystery'},
 {'title': 'A Murder in Time',
  'price': '£16.64',
  'star_rating': 'One',
  'availability': 'In stock',
  'category': 'Mystery'},
 {'title': 'The Murder of Roger Ackroyd (Hercule Poirot #4)',
  'price': '£44.10',
  'star_rating': 'Four',
  'availability': 'In stock',
  'category': 'Mystery'},
 {'title': 'The Last Mile (Amos Decker #2)',
  'price': '£54.21',
  'star_rating': 'Two',
  'availability': 'In stock',
  'category': 'Mystery'},
 {'title': 'That Darkness (Gardiner and Renner #1)',
  'price': '£13.92',
  'star_rating': 'One',
  'availability': 'In stock',
  'category': 'Mystery'},
 {

#Task2
Clean the scraped fields into proper types:

Strip the currency symbol from price and convert it to a float column price_gbp.

Convert the text star rating (One…Five) into an integer column rating (1–5).

Parse the availability text into a boolean column in_stock.

If any field fails to parse for a given row (e.g., unexpected text), handle it with the median-imputation approach for numeric fields or drop the row (state and justify your choice) — do not leave the pipeline crashing on messy rows.

In [64]:
#Task-2

df_books_details=pd.DataFrame(books_details)
print(df_books_details)

#Price conversion to float
df_books_details['price_gbp'] = pd.to_numeric(
    df_books_details['price'].astype(str).str.replace('£', '', regex=False).str.strip()
)

#Rating conversion to integer
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}
df_books_details['rating'] = df_books_details['star_rating'].map(rating_map)

#Availability conversion to boolean
df_books_details['in_stock'] = df_books_details['availability'].astype(str).str.lower().str.contains("in stock", na=False).astype(bool)

#Imputation Strategy for Messy Rows
if df_books_details["price_gbp"].isnull().any():
        med_price = df_books_details["price_gbp"].median()
        df_books_details["price_gbp"] = df_books_details["price_gbp"].fillna(med_price)

if df_books_details["rating"].isnull().any():
        mode_rating = df_books_details["rating"].mode()[0]
        df_books_details["rating"] = df_books_details["rating"].fillna(mode_rating).astype(int)


df_books_details


                                              title   price star_rating  \
0                                     Sharp Objects  £47.82        Four   
1                              In a Dark, Dark Wood  £19.63         One   
2                               The Past Never Ends  £56.50        Four   
3                                  A Murder in Time  £16.64         One   
4   The Murder of Roger Ackroyd (Hercule Poirot #4)  £44.10        Four   
..                                              ...     ...         ...   
75                           My Name Is Lucy Barton  £41.56         One   
76                                    My Mrs. Brown  £24.48       Three   
77            Mr. Mercedes (Bill Hodges Trilogy #1)  £28.90         One   
78                        I Am Pilgrim (Pilgrim #1)  £10.60        Four   
79                 Eligible (The Austen Project #4)  £27.09       Three   

   availability category  
0      In stock  Mystery  
1      In stock  Mystery  
2      In stock  M

,title,price,star_rating,availability,category,price_gbp,rating,in_stock
0,Sharp Objects,£47.82,Four,In stock,Mystery,47.82,4,True
1,"In a Dark, Dark Wood",£19.63,One,In stock,Mystery,19.63,1,True
2,The Past Never Ends,£56.50,Four,In stock,Mystery,56.50,4,True
3,A Murder in Time,£16.64,One,In stock,Mystery,16.64,1,True
4,The Murder of Roger Ackroyd (Hercule Poirot #4),£44.10,Four,In stock,Mystery,44.10,4,True
...,...,...,...,...,...,...,...,...
75,My Name Is Lucy Barton,£41.56,One,In stock,Fiction,41.56,1,True
76,My Mrs. Brown,£24.48,Three,In stock,Fiction,24.48,3,True
77,Mr. Mercedes (Bill Hodges Trilogy #1),£28.90,One,In stock,Fiction,28.90,1,True
78,I Am Pilgrim (Pilgrim #1),£10.60,Four,In stock,Fiction,10.60,4,True


In [41]:
df_books_details.info()

df_books_details.describe()

df_books_details.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         80 non-null     object 
 1   price         80 non-null     object 
 2   star_rating   80 non-null     object 
 3   availability  80 non-null     object 
 4   category      80 non-null     object 
 5   price_gbp     80 non-null     float64
 6   rating        80 non-null     int64  
 7   in_stock      80 non-null     bool   
dtypes: bool(1), float64(1), int64(1), object(5)
memory usage: 4.6+ KB


,0
title,0
price,0
star_rating,0
availability,0
category,0
price_gbp,0
rating,0
in_stock,0


#Task-3

Convert price_gbp to a price_inr column using the project's fixed baseline conversion rate: 1 GBP = 105.50 INR. This is an artificial, project-defined constant for this assignment, not a live or historical market rate, so it never needs a lookup or a date reference. This fixed-rate conversion is the required, keyless baseline and is what gets graded for this task — it requires no external API call and no network access; simply state this exact rate in your README. (Optional, ungraded stretch — must not affect your required submission: if you want extra practice with the requests library and explicit HTTP status-code handling, you may additionally look up any free, keyless currency-conversion API of your own choosing, check its response status code explicitly, and fall back to the fixed rate above on any failure. This is entirely optional; your price_inr column must be fully correct using only the required fixed-rate baseline, since that path alone is what gets graded.)

In [65]:
#Task-3

PRICE_GBP_TO_INR_RATE=105.50
#Convert GBP to INR using fixed rate constant (1 GBP = 105.50 INR)
df_books_details["price_inr"] = (df_books_details["price_gbp"] * PRICE_GBP_TO_INR_RATE).round(2)

df_books_details


,title,price,star_rating,availability,category,price_gbp,rating,in_stock,price_inr
0,Sharp Objects,£47.82,Four,In stock,Mystery,47.82,4,True,5045.01
1,"In a Dark, Dark Wood",£19.63,One,In stock,Mystery,19.63,1,True,2070.96
2,The Past Never Ends,£56.50,Four,In stock,Mystery,56.50,4,True,5960.75
3,A Murder in Time,£16.64,One,In stock,Mystery,16.64,1,True,1755.52
4,The Murder of Roger Ackroyd (Hercule Poirot #4),£44.10,Four,In stock,Mystery,44.10,4,True,4652.55
...,...,...,...,...,...,...,...,...,...
75,My Name Is Lucy Barton,£41.56,One,In stock,Fiction,41.56,1,True,4384.58
76,My Mrs. Brown,£24.48,Three,In stock,Fiction,24.48,3,True,2582.64
77,Mr. Mercedes (Bill Hodges Trilogy #1),£28.90,One,In stock,Fiction,28.90,1,True,3048.95
78,I Am Pilgrim (Pilgrim #1),£10.60,Four,In stock,Fiction,10.60,4,True,1118.30


#Task-4

Design a normalized SQLite schema with at least two tables sharing a primary/foreign key relationship, for example:

categories(category_id INTEGER PRIMARY KEY, category_name TEXT UNIQUE)
books(book_id INTEGER PRIMARY KEY, title TEXT, price_gbp REAL, price_inr REAL, rating INTEGER, in_stock INTEGER, category_id INTEGER REFERENCES categories(category_id))
(You may rename columns/tables, but the two-table PK/FK structure is required.)

In [66]:
#Task-4

with sqlite3.connect('data_base.db') as conn:
  cursor = conn.cursor()
  query='''
  CREATE TABLE IF NOT EXISTS categories(
  category_id INTEGER PRIMARY KEY AUTOINCREMENT,
  category_name TEXT UNIQUE NOT NULL
  )'''
  cursor.execute(query)
  conn.commit()

  query='''
  CREATE TABLE IF NOT EXISTS books(
  book_id INTEGER PRIMARY KEY AUTOINCREMENT,
  title TEXT NOT NULL,
  price_gbp REAL NOT NULL,
  price_inr REAL NOT NULL,
  rating INTEGER NOT NULL,
  in_stock INTEGER NOT NULL,
  category_id INTEGER,
  FOREIGN KEY (category_id) REFERENCES categories(category_id)
  )'''
  cursor.execute(query)
  conn.commit()
  print("All tables created successfully!")


All tables created successfully!


#Task-5

Using Python's sqlite3 (or pandas.DataFrame.to_sql), insert your cleaned, converted data into this schema. Then write and execute at least 5 SQL queries against the database that collectively demonstrate: SELECT/WHERE, ORDER BY, LIMIT, DISTINCT, and (IN or BETWEEN) — plus at least one JOIN between your two tables (e.g., "list the 10 highest-rated books per category"). Save each query string and its output.

In [67]:
#Task-5

with sqlite3.connect('data_base.db') as conn:
  cursor = conn.cursor()
  df_clean=df_books_details[['title','price_gbp','price_inr','rating','in_stock','category']].copy()

  # Insert unique categories
  unique_categories = sorted(df_clean["category"].unique())
  for cat in unique_categories:
        cursor.execute("INSERT INTO categories (category_name) VALUES (?)", (cat,))

  conn.commit()

  # Map category names to category_ids
  cat_map = {name: cat_id for cat_id, name in cursor.execute("SELECT category_id, category_name FROM categories")}

  # Insert books
  books_to_insert = []
  for _, row in df_clean.iterrows():
        books_to_insert.append((
            row["title"],
            row["price_gbp"],
            row["price_inr"],
            int(row["rating"]),
            1 if row["in_stock"] else 0,
            cat_map[row["category"]]
        ))

  cursor.executemany('''
    INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', books_to_insert)
  conn.commit()
  print("\nData Inserted successfully")

  query = "PRAGMA table_info(categories)"
  columns_info = pd.read_sql(query, conn)
  print(f"\nCategories Table:\n{columns_info[['name', 'type']]}\n")

  query = "PRAGMA table_info(books)"
  columns_info = pd.read_sql(query, conn)
  print(f"Books Table:\n{columns_info[['name', 'type']]}\n")

  cursor.execute("SELECT COUNT(*) FROM categories")
  row_count = cursor.fetchone()[0]
  print(f"Total rows in categories table: {row_count}")

  cursor.execute("SELECT COUNT(*) FROM books")
  row_count = cursor.fetchone()[0]
  print(f"Total rows in books table: {row_count}")



Data Inserted successfully

Categories Table:
            name     type
0    category_id  INTEGER
1  category_name     TEXT

Books Table:
          name     type
0      book_id  INTEGER
1        title     TEXT
2    price_gbp     REAL
3    price_inr     REAL
4       rating  INTEGER
5     in_stock  INTEGER
6  category_id  INTEGER

Total rows in categories table: 4
Total rows in books table: 80


In [68]:
with sqlite3.connect('data_base.db') as conn:
  cursor = conn.cursor()
  queries = {
        "Query 1 (SELECT/WHERE - Filter in-stock books under 20 GBP)": """
            SELECT title, price_gbp, in_stock
            FROM books
            WHERE price_gbp < 20.0 AND in_stock = 1;
        """,

        "Query 2 (ORDER BY & LIMIT - Top 5 most expensive books in INR)": """
            SELECT title, price_inr, rating
            FROM books
            ORDER BY price_inr DESC
            LIMIT 5;
        """,

        "Query 3 (DISTINCT - Unique star ratings available in dataset)": """
            SELECT DISTINCT rating
            FROM books
            ORDER BY rating ASC;
        """,

        "Query 4 (BETWEEN / IN - Books with rating 4 or 5 and price between 10 and 30 GBP)": """
            SELECT title, rating, price_gbp
            FROM books
            WHERE rating IN (4, 5) AND (price_gbp BETWEEN 10.0 AND 30.0)
            LIMIT 5;
        """,

        "Query 5 (JOIN - List top 10 rated books with Category Name)": """
            SELECT b.title, c.category_name, b.rating, b.price_gbp, b.price_inr
            FROM books b
            JOIN categories c ON b.category_id = c.category_id
            WHERE b.rating >= 4
            ORDER BY b.rating DESC, b.price_gbp DESC
            LIMIT 10;
        """
    }
  print("=== EXECUTING SQL QUERIES ===")
  results = {}
  for name, sql in queries.items():
        print(f"\n--- {name} ---")
        df_res = pd.read_sql_query(sql, conn)
        results[name] = df_res
        print(df_res.to_string(index=False))


=== EXECUTING SQL QUERIES ===

--- Query 1 (SELECT/WHERE - Filter in-stock books under 20 GBP) ---
                                                                            title  price_gbp  in_stock
                                                             In a Dark, Dark Wood      19.63         1
                                                                 A Murder in Time      16.64         1
                                           That Darkness (Gardiner and Renner #1)      13.92         1
                                             Tastes Like Fear (DI Marnie Rome #3)      10.69         1
                                          A Study in Scarlet (Sherlock Holmes #1)      16.73         1
                                                       Hide Away (Eve Duncan #20)      11.84         1
                                                                Playing with Fire      13.71         1
                                                                      Lilac G

#Task-6

Read back at least two of the above query results into pandas DataFrames using pd.read_sql(...), and separately reproduce the join-query's result using pd.merge(...) directly on your in-memory DataFrames (no SQL) — show that both approaches produce equivalent output.

In [69]:
#Task-6

with sqlite3.connect('data_base.db') as conn:
  cursor = conn.cursor()
  # 1. Fetch via SQL JOIN (pd.read_sql)
  join_sql = """
        SELECT b.title, c.category_name, b.rating, b.price_gbp, b.price_inr
        FROM books b
        JOIN categories c ON b.category_id = c.category_id
        WHERE b.rating >= 4
        ORDER BY b.rating DESC, b.price_gbp DESC
        LIMIT 10;
    """
  sql_result = pd.read_sql_query(join_sql, conn)

    # 2. Replicate via in-memory pd.merge
  categories_df = pd.read_sql_query("SELECT * FROM categories", conn)
  books_df = pd.read_sql_query("SELECT * FROM books", conn)

  merged_df = (
        pd.merge(books_df, categories_df, on="category_id")
        .query("rating >= 4")
        .sort_values(by=["rating", "price_gbp"], ascending=[False, False])
        [['title', 'category_name', 'rating', 'price_gbp', 'price_inr']]
        .head(10)
        .reset_index(drop=True)
    )

  print("\nSQL Result (pd.read_sql):")
  print(sql_result.to_string(index=False))

  print("\nPandas Result (pd.merge):")
  print(merged_df.to_string(index=False))

  # Compare equivalence
  is_equal = sql_result.equals(merged_df)
  print(f"\nExact Output Match? -> {is_equal}")


SQL Result (pd.read_sql):
                                                                   title      category_name  rating  price_gbp  price_inr
                                 A Flight of Arrows (The Pathfinders #2) Historical Fiction       5      55.53    5858.42
                                Finders Keepers (Bill Hodges Trilogy #2)            Fiction       5      53.53    5647.42
The Bachelor Girl's Guide to Murder (Herringford and Watts Mysteries #1)            Mystery       5      52.30    5517.65
                 Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)     Sequential Art       5      52.29    5516.60
                                    The Regional Office Is Under Attack!            Fiction       5      51.36    5418.48
                                            We Love You, Charlie Freeman            Fiction       5      50.27    5303.48
                                  A Time of Torment (Charlie Parker #14)            Mystery       5      48.35    5100.